In [37]:
import pandas as pd
from tqdm import tqdm
from itables import show as table_show

In [38]:
ENTREPOT_PATH = '~/Bureau/utils/data/'
METEO_PATH = '~/Bureau/utils/data/meteo/'
SPATIAL_PATH = './data/spatial/'

In [39]:
df = {}

def import_df(df_name, path_data, sep, index_col=None):
    df[df_name] = pd.read_csv(path_data+df_name+'.csv', sep = sep, index_col=index_col, low_memory=False)

def import_dfs(df_names, path_data, sep = ',', index_col=None, verbose=False):
    for df_name in tqdm(df_names) : 
        if(verbose) :
            print(" - ", df_name)
        import_df(df_name, path_data, sep, index_col=index_col)

tables_entrepot = [
        'sdc', 
        'dispositif',
        'synthetise',
        'connection_synthetise',
        'noeuds_synthetise',
        'noeuds_realise',
        'parcelle',
        'zone', 
        'noeuds_synthetise_restructure', 
]

tables_performances = [
        'itk_realise_performance',
        'itk_synthetise_performance',
        'typologie_can_culture'

]

# import des données de l'entrepôt avec la colonne 'id' en index 
import_dfs(tables_entrepot, ENTREPOT_PATH, sep = ',',index_col='id',verbose=False) 
import_dfs(tables_performances, ENTREPOT_PATH, sep = ',',verbose=False) 

100%|██████████| 3/3 [00:21<00:00,  7.11s/it]


In [40]:
realise_studied_node_ids = [
    'fr.inra.agrosyst.api.entities.effective.EffectiveCropCycleNode_73e06d01-07f7-4268-9cca-bcd7d3518808',
    'fr.inra.agrosyst.api.entities.effective.EffectiveCropCycleNode_3c555994-c3a0-47c2-b618-17a026beb21f',
    'fr.inra.agrosyst.api.entities.effective.EffectiveCropCycleNode_214ac958-3444-4308-9647-4dd79fac5c88',
    'fr.inra.agrosyst.api.entities.effective.EffectiveCropCycleNode_f63d381c-a2dd-46dd-a437-dff786b051bc',
    'fr.inra.agrosyst.api.entities.effective.EffectiveCropCycleNode_21e99185-c2f6-45ea-9524-922009aba0be', 
    'fr.inra.agrosyst.api.entities.effective.EffectiveCropCycleNode_d3bbd8e4-2f83-431b-906c-4e0a3cad88a8',
    'fr.inra.agrosyst.api.entities.effective.EffectiveCropCycleNode_a6c4beef-398e-48a7-ad8c-118b06f72c6d'
]

synthetise_studied_connexion_ids = [
    'fr.inra.agrosyst.api.entities.practiced.PracticedCropCycleConnection_b2fe31af-208e-4c25-ab62-f1997ea83d35',
    'fr.inra.agrosyst.api.entities.practiced.PracticedCropCycleConnection_09dbc1d4-11e6-4fd5-ba4f-a6c82c626e72',       'fr.inra.agrosyst.api.entities.practiced.PracticedCropCycleConnection_013ce299-b982-4f4b-bdd7-f2a0b9aee1f5',
    'fr.inra.agrosyst.api.entities.practiced.PracticedCropCyclePhase_89893b4d-c5a2-4df4-8fa0-74dd6227e9a0',
    'fr.inra.agrosyst.api.entities.practiced.PracticedCropCycleConnection_05d9344c-0e80-43e1-9f5c-4311965f8810'
]


In [41]:
# en synthétisé
df['connection_synthetise_test'] = df['connection_synthetise'].loc[
    df['connection_synthetise'].index.isin(synthetise_studied_connexion_ids)
]
df['noeuds_synthetise_test'] = df['noeuds_synthetise'].loc[
    df['noeuds_synthetise'].index.isin(df['connection_synthetise_test']['cible_noeuds_synthetise_id'])
]
df['noeuds_synthetise_restructure_test'] = df['noeuds_synthetise_restructure'].loc[
    df['noeuds_synthetise_restructure'].index.isin(df['noeuds_synthetise_test'].index)
]
df['synthetise_test'] = df['synthetise'].loc[
    df['synthetise'].index.isin(df['noeuds_synthetise_test']['synthetise_id'])
]
df['sdc_synthetise_test'] = df['sdc'].loc[
    df['sdc'].index.isin(df['synthetise_test']['sdc_id'])
]

df['itk_synthetise_performance_test'] = df['itk_synthetise_performance'].loc[
    df['itk_synthetise_performance']['connection_synthetise_id'].isin(synthetise_studied_connexion_ids)
]

# en réalisé
df['noeuds_realise_test'] = df['noeuds_realise'].loc[
    df['noeuds_realise'].index.isin(realise_studied_node_ids)
]
df['zone_test'] = df['zone'].loc[
    df['zone'].index.isin(df['noeuds_realise_test']['zone_id'])
]
df['parcelle_test'] = df['parcelle'].loc[
    df['parcelle'].index.isin(df['zone_test']['parcelle_id'])
]
df['sdc_realise_test'] = df['sdc'].loc[
    df['sdc'].index.isin(df['parcelle_test']['sdc_id'])
]

df['itk_realise_performance_test'] = df['itk_realise_performance'].loc[
    (df['itk_realise_performance']['noeuds_realise_id'].isin(realise_studied_node_ids))
]

# commun
df['typologie_can_culture_test'] = df['typologie_can_culture'].loc[
    (df['typologie_can_culture']['culture_id'].isin(df['noeuds_synthetise_restructure_test']['culture_id'].values)) |
    (df['typologie_can_culture']['culture_id'].isin(df['noeuds_realise_test']['culture_id'].values))
]


In [42]:
path='./'
df['zone_test'].to_csv(path+'zone.csv')
df['parcelle_test'].to_csv(path+'parcelle'+'.csv')
df['noeuds_realise_test'].to_csv(path+'noeuds_realise'+'.csv')
df['sdc_realise_test'].to_csv(path+'sdc'+'.csv')
df['synthetise_test'].to_csv(path+'synthetise'+'.csv')
df['noeuds_synthetise_test'].to_csv(path+'noeuds_synthetise'+'.csv')
df['noeuds_synthetise_restructure_test'].to_csv(path+'noeuds_synthetise_restructure'+'.csv')
df['connection_synthetise_test'].to_csv(path+'connection_synthetise'+'.csv')
df['itk_realise_performance_test'].to_csv(path+'itk_realise_performance'+'.csv')
df['itk_synthetise_performance_test'].to_csv(path+'itk_synthetise_performance'+'.csv')
df['typologie_can_culture_test'].to_csv(path+'typologie_can_culture'+'.csv')